# Retriever
- 비정형 질의(query)를 입력 받아 Vector store에서 관련된 내용을 검색하는 기능을 제공하는 인터페이스로 다양한 데이터 소스에서 정보를 검색하여 대규모 언어 모델(LLM) 기반 애플리케이션의 **정확성을** 향상시키는 데 핵심적인 역할을 한다.

![RAG](figures/rag2.png)

## 주요 특징
- **다양한 데이터 소스 지원**
	- Retriever는 벡터 스토어, 그래프 데이터베이스, 관계형 데이터베이스 등 여러 종류의 검색 시스템과 상호작용할 수 있는 통일된 인터페이스를 제공한다다.
- **간단한 인터페이스**: Retriever는 문자열 형태의 쿼리를 입력받아 관련 문서의 리스트를 반환한다. 이러한 단순한 구조 덕분에 다양한 검색 시스템과 쉽게 통합할 수 있다. 

### VectorStore와 Retriever

VectorStore와 Retriever는 모두 Vector Database 검색(Retrieval)과 관련되지만, 역할과 책임이 다르다.
- **VectorStore**
  - VectorStore는 벡터 데이터를 관리하는 저장소로 다음과 같은 기능을 직접 제공한다.

  	- 저장: 문서를 벡터(임베딩)로 변환해 데이터베이스에 저장
  	- 삭제/수정: 저장된 벡터를 지우거나 업데이트
  	- 인덱싱: 빠르게 검색할 수 있도록 벡터 인덱스를 생성·관리
  	- 유사도 검색: 입력 질문과 가장 비슷한 벡터를 찾아 관련 문서를 반환
  - 즉, VectorStore는 **벡터 데이터 관리 + 벡터 검색 기능을 모두 포함한 인터페이스**이다.
  - 내부적으로는 실제 벡터 DB(예: FAISS, Chroma, Pinecone, Milvus 등)와 연결되어 있고, 그 DB를 직접 다루는 역할을 한다.

- **Retriever**
	- Retriever는 "검색 요청을 어떻게 할지"만 정리해 둔 실행 모듈로 직접 벡터를 저장하거나 인덱스를 만들지는 않는다.
	- 검색 개수(k), 메타데이터 필터, 하이브리드 검색 여부, 재랭킹 적용 여부 등 설정된 전략에 따라 VectorStore(또는 다른 검색 백엔드)에 검색을 요청하고 결과만 받아서 반환한다.
	- Runnable 타입이므로 체인(LCEL) 안에 포함될 수 있다.

## 다양한 Retriever 방식

- Retriever의 검색 성능을 높이기 위한 다양한 Retriever의 확장 전략이 있다. 

1. **벡터 스토어(Vector Store) Retriever**
   - VectorStore로 부터 유사도를 기반으로 검색하는 **가장 기본 Retriever**
   - 텍스트 조각(청크)마다 **임베딩(embedding)을** 생성하여 벡터 공간에 저장하고, 쿼리 임베딩과의 **코사인 유사도(cosine similarity)** 등을 기반으로 유사한 텍스트를 검색한다.
   - 검색 속도가 빠르고 구현이 간단하여, 기본적인 검색 시스템을 구축할 때 적합하다.
2. **[ParentDocumentRetriever](https://python.langchain.com/docs/how_to/parent_document_retriever/)**
   - 하나의 문서를 여러 청크로 나눈 뒤 각각을 인덱싱하고, 쿼리와 가장 유사한 청크를 찾은 다음 해당 청크가 속한 **전체 원본 문서**를 반환한다.
   - 작은 정보 조각들이 모여 하나의 문서를 구성할 때 유용하며, 문맥을 넓게 유지할 수 있다.
3. **[MultiVectorRetriever](https://python.langchain.com/docs/how_to/multi_vector/)**
   - 각 문서에 대해 요약을 하거나, 가상의 질문을 생성하거나, 사람이 중요한 내용을 직접 추가하여 문서당 여러 개의 임베딩 벡터를 생성한다.
   - 텍스트 전체보다 더 핵심적인 정보가 검색에 반영되도록 하고자 할 때 효과적이다.
   - 특히, 문서가 길거나, 중요한 내용이 문서의 특정 부분에 집중되어 있는 경우에 유리하다.
4. **[SelfQueryRetriever](https://python.langchain.com/docs/how_to/self_query/)**
   - 대규모 언어 모델(LLM, Large Language Model)을 활용하여 사용자의 질문을 적절한 검색어와 **메타데이터(metadata)** 필터로 자동 변환한다.
   - 예를 들어, 문서의 작성자, 날짜, 태그와 같은 메타데이터를 기준으로 검색할 수 있다.
   - 문서 자체의 내용뿐만 아니라, 문서에 부가된 속성 정보에 대해 질문할 때 유용하다.
5. **[ContextualCompressionRetriever](https://python.langchain.com/docs/how_to/contextual_compression/)**
   - 기존 Retriever와 조합되어 사용된다.
   - 먼저 일반적인 검색을 수행한 후, 검색된 문서들에서 쿼리와 관련 없는 불필요한 정보를 제거하고 핵심 내용만을 추출하여 반환한다.
   - 정보를 요약하거나 압축하여 LLM에 전달할 문서 길이를 줄일 때 유용하다.
6. **[MultiQueryRetriever](https://python.langchain.com/docs/how_to/MultiQueryRetriever/)**
   - LLM을 이용해 하나의 쿼리로부터 여러 가지 변형된 쿼리를 생성하고, 각 쿼리에 대해 검색을 수행한 뒤 결과를 합치는 방식.
   - 다양한 표현에 강해 검색 범위를 넓히고 성능을 높인다.
7. **[EnsembleRetriever](https://python.langchain.com/docs/how_to/ensemble_retriever/)**
   - 여러 개의 Retriever(예: BM25, 벡터 기반 등)를 결합해 가중치를 기반으로 결과를 조합(re-ranking)한다.
   - 서로 다른 장점을 가진 Retriever를 하나로 묶어 성능을 강화한다.

In [ ]:
# Retriever 생성
## vectorstore.as_retriever() 

In [1]:
###################################
# data/olympic.md 저장 -> 검색
###################################
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, SparseIndexParams

from dotenv import load_dotenv

load_dotenv()

C:\Users\Playdata\AppData\Local\Temp\ipykernel_10616\2703826370.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


True

In [2]:
########################
# 문서 로드 및 split
########################
loader = TextLoader("data/olympic_wiki.md", encoding="utf-8")
documents = loader.load()
doc_txt = '\n\n'.join(doc.page_content for doc in documents)
# print(doc_txt)

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "H1"), ("##", "H2"), ("###", "H3")
    ]
)
docs = splitter.split_text(doc_txt)
print(len(docs))


25


In [3]:
##################################
# Embedding -> Store
##################################

# 연결 + collection 생성
COLLECTION_NAME = "olympic_info"
VECTOR_SIZE = 3072

client = QdrantClient(url="http://localhost:6333")

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense":VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse":SparseVectorParams(index=SparseIndexParams(on_disk=False))
    }
)

True

In [4]:
#########################################################
# VectorStore 생성 -> Dataset upsert
#########################################################
dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
sparse_embeddings = FastEmbedSparse(model_name="qdrant/bm25")

vectorstore = QdrantVectorStore(
    client=client, 
    collection_name=COLLECTION_NAME,
    # embedding 모델들
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    # vector 저장소 지정
    vector_name="dense",
    sparse_vector_name="sparse",
    # 검색방식 - HYBRID, DENSE(기본), SPARSE
    retrieval_mode=RetrievalMode.HYBRID
)
# 저장
vectorstore.add_documents(documents=docs)

# metadata에 대한 Index 를 설정.

['347df1fa3b51434eac4913f0be2e2eda',
 '40b13ffa515b4d9eaec4d17043398d73',
 'c48efe599ef04805ad254bd9cb746e79',
 '1202cd83e1564450afef918b9f313440',
 '72fa447e0a53470089c990d23712c5ef',
 '1c3ca1c227384c96b24db618d4d1ecb0',
 'f8aa0a5f7e554a40b02898cbfbc09200',
 '6af2a123d88f44e8a8e4d139b577522f',
 'b52dc2dec1664ed4aa80561bb17117d5',
 'bc81c891ecb8406181ec711ba95cc3bf',
 'ee3207cfd3d84be588a1d70f6a5caf28',
 'ca63fd1a6f93436c83017ac5d0add004',
 '4e25815ac1c54e65bc63ca217165f875',
 '6b0262a393ad4beebdb42b8d997f5854',
 '9f2dfb1b5c104145b3961b799c77d2a5',
 '45cb5f7052c348e389192a541306400c',
 '0d83e7d0c29d4173968b1756f42d484f',
 '5ffc2a4f79b54a47943c19bda5ea3f3b',
 '5417d8aba0354210878527ee064724a5',
 '9a524aa2d6054653a909e1c2e68fa5b4',
 'cf64109a2a804faa98feab4159aecce2',
 '2957d3a8ce4b4ba49d1ffbe86ebe7cc7',
 '07a9024149fc47019d85005c5b743109',
 '90e2aa4ca2ab47a282ea2d0fbc2ff3ec',
 '5ab9e27347364d05bcfc9c52b7d1b725']

In [5]:
##################################
# vector store로 부터 Retiever 생성
##################################
retriever = vectorstore.as_retriever()

In [6]:
from langchain_core.runnables import Runnable

print(isinstance(vectorstore, Runnable), isinstance(retriever, Runnable))

query = "올림픽에서 일어난 논란들을 알려줘."
result = retriever.invoke(query)
result

False True


[Document(metadata={'H1': '올림픽', 'H2': '논란', 'H3': '정치', '_id': '9a524aa2-d605-4653-a909-e1c2e68fa5b4', '_collection_name': 'olympic_info'}, page_content="쿠베르탱이 말했던 원래 이념과는 반대로 올림픽이 정치 혹은 체제 선전의 장으로 이용되는 경우가 있었다. 1936년 하계 올림픽을 개최할 때 당시의 나치독일은 나치는 자비롭고 평화를 위한다는 것을 설명하고 싶어했다. 또 이 올림픽에서 아리안족의 우월함을 보여줄 생각이었으나 이는 흑인이었던 제시 오언스가 금메달을 4개나 따내면서 실현되지는 못했다. 소련은 헬싱키에서 열린 1952년 하계 올림픽 때 처음으로 참가했다. 그 전에는 소련이 조직한 스파르타키아다라는 대회에 1928년부터 참가했었다. 다른 공산주의 국가들은 1920년대와 1930년대의 전쟁 기간 사이에 노동자 올림픽(Socialist Workers' Sport International)을 조직했는데, 이는 올림픽을 자본가와 귀족들의 대회로 여기고 그에 대한 대안으로 고안된 대회였다. 그 이후 소련은 1956년 하계 올림픽부터 1988년 하계 올림픽까지 엄청난 스포츠강국의 면모를 보여주며 올림픽에서의 명성을 드높였다.  \n선수 개인이 자신의 정치적 성향에 대해 표현하기도 했다. 멕시코 시티에서 열린 1968년 하계 올림픽의 육상부문 200m 경기에서 각각 1위와 3위를 한 미국의 토미 스미스와 존 카를로스는 시상식 때 블랙 파워 설루트(Black Power salute , 흑인 차별 반대 행위)를 선보였으며 2위를 한 피터 노먼도 상황을 깨닫고 스미스와 카를로스의 행위를 지지한다는 뜻에서 급하게 인권을 위한 올림픽 프로젝트(OPHR) 배지를 달았다. 이 사건에 대해서 IOC 위원장이었던 에이버리 브런디지는 미국 올림픽 위원회에 이 두 선수를 미국으로 돌려보내거나 미국 육상팀 전부를 돌려보내는 둘 중 하나의 선택을 하게 했고, 미국 올

In [7]:
# Chain 구성: retriever -> prompt template -> llm -> output parser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# PromptTempate 구성 + LLM 모델
prompt = ChatPromptTemplate.from_template(
    template="""<instruction>
당신은 친절한 QA Assistant입니다.
질문에 대해 주어진 context를 기반으로 답을 해주세요.
context에 질문과 관련된 내용이 없을 경우 "주어진 정보로는 답을 알 수 없습니다." 라고 답을 하세요.
context에 없는 내용으로 답을 만들지 마세요.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
<output-format>
- 답변을 context의 어느부분을 참조하였는지 각주를 달아주세요.
- 답변은 markdown 형식으로 작성하세요.
- 답변 구성은 example을 참조하세요.
<example>
# 답변 
최종 답변내용

# 참조내용
context에서 참조한 내용
</example>
</output-format>"""
)
model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

def format_str(docs:list) -> str:
    """
    retriever가 찾은 문서 리스트(list[Document])에서 
        page_content만 추출해서 하나의 문자열로 합쳐서 반환하는 함수.
    Prompt Template의 context에 넣어줄 값들만 str로 만든다.
    """
    return "\n\n".join(doc.page_content for doc in docs)

# 1. RunnableParallel{retriever,RunnablePassthrough}
# 문서검색(Retriever) -> PromptTemplate -> llm -> output parser
chain = {
    "context":retriever | format_str,     # retriever 처리결과
    "query":RunnablePassthrough() # 입력받은 질문을 전달
} | prompt | model | parser

In [8]:
query = "IOC는 어떤 단체인가요?"
query = "동계올림픽에 대해 알려줘."
response = chain.invoke(query)

In [9]:
print(response)

# 답변
동계올림픽은 얼음과 눈을 이용한 경기 종목을 다루는 올림픽으로, 20세기에 올림픽 운동이 발전하면서 새롭게 포함되었습니다.[^1] 올림픽은 하계 올림픽과 동계 올림픽이 2년마다 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독합니다.[^2]

# 참조내용
[^1]: “이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽 … 등을 들 수 있다.”
[^2]: “올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다.”


In [13]:
#####################################################
# Retriever를 생성시 다양한 정보를 설정하는 방법
# as_retriever(옵션-파라미터들)
#####################################################

# 조회방식 설정
## search_type:  similarity_score_threshold - score_threshold를 지정. (검색될 문서의 최소 유사도 점수)
##               mmr (MMR 알고리즘 적용-유사도:다양성)

# search_kwargs=dict. 다양한 설정들을 지정. (similarity_search() 호출시 지정하는 파라미터들)
retriever2 = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k":7,
        "score_threshold":0.7
    }
)
result = retriever2.invoke(query)
result

[Document(metadata={'H1': '올림픽', 'H2': '논란', 'H3': '보이콧', '_id': '5417d8ab-a035-4210-8785-27ee064724a5', '_collection_name': 'olympic_info'}, page_content='올림픽에서 첫 번째 보이콧은 1956년 하계 올림픽에서 시작되었다. 네덜란드, 스페인, 스위스는 소련의 헝가리 침공에 항의해 참가를 거부했다. 캄보디아, 이집트, 이라크, 레바논은 제2차 중동 전쟁 때문에 보이콧했다. 1972년과 1976년 올림픽에는 많은 아프리카의 국가들이 남아프리카 공화국과 로디지아에서 일어나는 인종 차별정권에 대한 항의의 표시로 올림픽 참가를 거부했다. 이 보이콧에는 뉴질랜드도 관계가 되어있는데, 뉴질랜드 럭비 국가 대표팀이 당시 아파르트헤이트정책을 쓰던 남아프리카 공화국과 경기를 했음에도 불구하고 뉴질랜드의 올림픽 참가가 허용되었기 때문이었다. 국제 올림픽 위원회는 이 두 보이콧에 대해 심각하게 고민했으나 후자의 뉴질랜드의 경우는 럭비가 올림픽 종목이 아니라는 이유를 내세워 뉴질랜드의 올림픽 참가 금지 요청을 거부했다. 당시 아프리카에 속해 있던 20개국과 가이아나, 이라크는 경기를 끝낸 선수들이 있었지만 탄자니아가 이끄는 올림픽 보이콧에 가세했다. 중화민국(타이완)도 1976년 몬트리올 올림픽 참가를 보이콧했는데, 그 이유는 중화인민공화국(중국)이 몬트리올 올림픽 조직위원회에게 타이완을 \'중화민국\'의 이름으로 참가하지 못하도록 압박을 가했기 때문이다. 타이완은 이것에 반발해서 중화민국의 국기와 중화민국의 국가를 계속 쓸 것이라고 밝혔다. 타이완은 1984년까지 올림픽에 참가하지 않았으며 그 후 참가할 때는 매화기와 특별한 찬가를 사용한다. 1980년과 1984년 올림픽 때는 냉전의 당사국들이 각각 반대진영에서 개최된 올림픽에 불참했다. 1980년에 열린 모스크바 올림픽 때는 소련의 아프가니스탄 침공에 대한 항의의 표시로 미국을 비롯한 65개국이 불참해서 1

In [14]:
# search_type: mmr
retriever3 = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "fetch_k":15,
        "lambda_mult":0.5,
        "k":5
    }
)
retriever3.invoke(query)

[Document(metadata={'H1': '올림픽', '_id': '347df1fa-3b51-434e-ac49-13f0be2e2eda', '_collection_name': 'olympic_info'}, page_content='올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.  \n또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변

In [ ]:
##########################################################
# payload filter - search_kwargs에 "filter":Filter객체
#########################################################
from qdrant_client.models import Filter, FieldCondition, MatchValue

filter_condition = Filter(
    should=[
        FieldCondition(key="metadata.H1", match=MatchValue(value="동계 올림픽")),
        FieldCondition(key="metadata.H2", match=MatchValue(value="동계 올림픽")),
        FieldCondition(key="metadata.H3", match=MatchValue(value="동계 올림픽")),
    ]
)

retriever4 = vectorstore.as_retriever(
    search_kwargs={
        "k":5,
        "filter":filter_condition
    }
)
retriever4.invoke(query)

[Document(metadata={'H1': '올림픽', '_id': '347df1fa-3b51-434e-ac49-13f0be2e2eda', '_collection_name': 'olympic_info'}, page_content='올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.  \n또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변

In [ ]:
####################################################################################
# search_type이나 search_kwargs 를 호출(invoke)할 때 동적으로 변경
####################################################################################
from langchain_core.runnables import ConfigurableField
retriever5 = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold":0.1}
)
# 리트리버의 어떤 설정(search_type, search_kwargs)을 호출할 때 동적으로 변경할 수 있는지를 지정.
retriever5 = retriever5.configurable_fields(
    # 바꿀파라미터변수=ConfigurableField() 객체
    search_kwargs=ConfigurableField(
        id="search_kwargs", # invoke() 호출때 변경할 값에 붙여줄 key값.
    ),
    # search_type=ConfigurableField(id="search_type")
)

In [ ]:
retriever5.invoke("동계 올림픽 종목에는 무엇이 있나요?") # 기본설정으로 조회

[Document(metadata={'H1': '올림픽', 'H2': '근대 올림픽', 'H3': '동계 올림픽', '_id': '72fa447e-0a53-4700-89c9-90d23712c5ef', '_collection_name': 'olympic_info'}, page_content='동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 실행하기 불가능한 종목들로 구성되어 있다. 피겨스케이팅, 아이스하키는 각각 1908년과 1920년에 하계올림픽 종목으로 들어가 있었다. IOC는 다른 동계 스포츠로 구성된 새로운 대회를 만들고 싶어 했고, 로잔에서 열린 1921년 올림픽 의회에서 겨울판 올림픽을 열기로 합의했다. 1회 동계올림픽은 1924년, 프랑스의 샤모니에서 11일간 진행되었고, 16개 종목의 경기가 치러졌다. IOC는 동계 올림픽이 4년 주기로 하계 올림픽과 같은 년도에 열리도록 했다. 이 전통은 프랑스의 알베르빌에서 열린 1992년 올림픽 때까지 지속되었으나, 노르웨이의 릴레함메르에서 열린 1994년 올림픽부터 동계 올림픽은 하계 올림픽이 끝난지 2년후에 개최하였다.'),
 Document(metadata={'H1': '올림픽', 'H2': '올림픽 경기 종목', '_id': '0d83e7d0-c29d-4173-968b-1756f42d484f', '_collection_name': 'olympic_info'}, page_content="올림픽 경기 종목은 총 33개부문 52개 종목에서 약 400개의 경기로 이루어져있다. 예를 들어서 하계 올림픽 부문인 레슬링은 자유형과 그레코로만형의 두 종목으로 나뉜다. 여기에서 10경기는 남자부, 4경기는 여자부로 열리며 분류기준은 체중이다. 하계 올림픽은 26개, 동계 올림픽은 7개 부문으로 이루어져있다. 하계 올림픽에서는 육상, 수영, 펜싱, 체조가 1회 대회때부터 한번도 빠짐없이 정식종목이었으며, 동계 올림픽에서는 크로스컨트리, 피겨 스케이팅, 아이스 하키, 노르딕 복합, 스키 점

In [ ]:
config = {
    "configurable":{
        "search_kwargs": {    # key: ConfigurableField에서 지정한 id값을 key로 지정.
            "k":10,
            "score_threshold":0.7   
        }
    }
}
retriever5.invoke("동계 올림픽 종목에는 무엇이 있나요?", config=config)
# config: RunnableConfig
# Runnable을 invoke() 할 때 전달하는 설정값.

[Document(metadata={'H1': '올림픽', 'H2': '근대 올림픽', 'H3': '동계 올림픽', '_id': '72fa447e-0a53-4700-89c9-90d23712c5ef', '_collection_name': 'olympic_info'}, page_content='동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 실행하기 불가능한 종목들로 구성되어 있다. 피겨스케이팅, 아이스하키는 각각 1908년과 1920년에 하계올림픽 종목으로 들어가 있었다. IOC는 다른 동계 스포츠로 구성된 새로운 대회를 만들고 싶어 했고, 로잔에서 열린 1921년 올림픽 의회에서 겨울판 올림픽을 열기로 합의했다. 1회 동계올림픽은 1924년, 프랑스의 샤모니에서 11일간 진행되었고, 16개 종목의 경기가 치러졌다. IOC는 동계 올림픽이 4년 주기로 하계 올림픽과 같은 년도에 열리도록 했다. 이 전통은 프랑스의 알베르빌에서 열린 1992년 올림픽 때까지 지속되었으나, 노르웨이의 릴레함메르에서 열린 1994년 올림픽부터 동계 올림픽은 하계 올림픽이 끝난지 2년후에 개최하였다.'),
 Document(metadata={'H1': '올림픽', 'H2': '올림픽 경기 종목', '_id': '0d83e7d0-c29d-4173-968b-1756f42d484f', '_collection_name': 'olympic_info'}, page_content="올림픽 경기 종목은 총 33개부문 52개 종목에서 약 400개의 경기로 이루어져있다. 예를 들어서 하계 올림픽 부문인 레슬링은 자유형과 그레코로만형의 두 종목으로 나뉜다. 여기에서 10경기는 남자부, 4경기는 여자부로 열리며 분류기준은 체중이다. 하계 올림픽은 26개, 동계 올림픽은 7개 부문으로 이루어져있다. 하계 올림픽에서는 육상, 수영, 펜싱, 체조가 1회 대회때부터 한번도 빠짐없이 정식종목이었으며, 동계 올림픽에서는 크로스컨트리, 피겨 스케이팅, 아이스 하키, 노르딕 복합, 스키 점

: 